# Structured Metadata Filtering [Step 2 - SQL-like Queries on Metadata]

> **MLCourse - Agentic AI - Vectorless RAG**

This notebook extends page-based retrieval with rich structured metadata.
We tag every chunk with multiple fields (chapter, section, page, word count,
has_figure, has_table, etc.) and run SQL-like filter queries against them.
This approach replaces vector search when the document has strong structure.

### Import all libraries needed for this notebook.


In [ ]:
import re                              # Regex for pattern detection
import json                            # Metadata serialization
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path               # File path handling
from pypdf import PdfReader            # PDF text extraction
from dataclasses import dataclass, field, asdict  # Structured metadata
from typing import List, Dict, Any     # Type hints


### Part 1: Configuration


In [ ]:
PDF_PATH = r"D:\projects\python\MLCourse\03_agentic_ai\data\attention_is_all_you_need.pdf"
CHUNK_SIZE = 600
OVERLAP = 100

print(f"PDF path: {PDF_PATH}")


### Part 2: Define the Chunk Metadata Schema


In [ ]:
# A dataclass makes the metadata structure explicit and self-documenting.

@dataclass
class ChunkMetadata:
    """Structured metadata for a single text chunk."""
    chunk_id: int = 0
    page: int = 0
    section: str = ""
    chapter: str = ""
    text: str = ""
    char_start: int = 0
    char_end: int = 0
    word_count: int = 0
    has_figure_ref: bool = False       # References a figure
    has_table_ref: bool = False        # References a table
    has_equation: bool = False         # Contains math notation
    has_citation: bool = False         # Contains [N] citations
    is_first_page_of_section: bool = False
    contains_author_names: bool = False

print("ChunkMetadata schema defined with 14 fields")


### Part 3: Extract and Build Metadata


In [ ]:
# We process the PDF and compute all metadata fields for each chunk.

reader = PdfReader(PDF_PATH)
total_pages = len(reader.pages)

# Known section boundaries in the Transformer paper.
SECTION_HEADINGS = {
    0: "Title",
    1: "Introduction",
    2: "Background",
    3: "Model Architecture",
    5: "Why Self-Attention",
    6: "Training",
    7: "Results",
    9: "Conclusion",
    10: "References",
    12: "Attention Visualizations",
}

# Extract raw text per page.
pages_raw = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    text = text.replace("\u2217", "*").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u0142", "l")
    text = text.replace("\u0105", "a")
    pages_raw.append({"page": i, "text": text})

# Author names commonly found in the paper.
AUTHOR_NAMES = [
    "vaswani", "shazeer", "parmar", "uszkoreit", "jones",
    "gomez", "kaiser", "polosukhin",
]

def detect_metadata_features(text):
    """Detect structural features in a chunk of text."""
    return {
        "has_figure_ref": bool(re.search(r"Figure\s+\d", text)),
        "has_table_ref": bool(re.search(r"Table\s+\d", text)),
        "has_equation": bool(re.search(r"[=∑∏∫≤≥∂∇]", text)),
        "has_citation": bool(re.search(r"\[\d+\]", text)),
        "contains_author_names": any(
            name in text.lower() for name in AUTHOR_NAMES
        ),
    }

# Build all chunks with metadata.
all_chunks: List[ChunkMetadata] = []
current_section = "Title"

for page_info in pages_raw:
    pg = page_info["page"]
    text = page_info["text"]

    # Update section if a new heading is found on this page.
    if pg in SECTION_HEADINGS:
        current_section = SECTION_HEADINGS[pg]

    # Chunk the page text.
    start = 0
    first_chunk_of_section = True
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end]
        features = detect_metadata_features(chunk_text)

        chunk = ChunkMetadata(
            chunk_id=len(all_chunks),
            page=pg,
            section=current_section,
            chapter="",  # No explicit chapters in this paper.
            text=chunk_text,
            char_start=start,
            char_end=end,
            word_count=len(chunk_text.split()),
            has_figure_ref=features["has_figure_ref"],
            has_table_ref=features["has_table_ref"],
            has_equation=features["has_equation"],
            has_citation=features["has_citation"],
            is_first_page_of_section=first_chunk_of_section and start == 0,
            contains_author_names=features["contains_author_names"],
        )
        all_chunks.append(chunk)
        first_chunk_of_section = False
        start += CHUNK_SIZE - OVERLAP

print(f"Built {len(all_chunks)} chunks with structured metadata")
print(f"Sections found: {sorted(set(c.section for c in all_chunks))}")


### Part 4: Metadata Statistics


In [ ]:
# Before querying, let us understand the metadata distribution.

sections = {}
for c in all_chunks:
    sections.setdefault(c.section, 0)
    sections[c.section] += 1

print("Chunks per section:")
for sec, count in sorted(sections.items()):
    print(f"  {sec}: {count} chunks")

figure_chunks = sum(1 for c in all_chunks if c.has_figure_ref)
table_chunks = sum(1 for c in all_chunks if c.has_table_ref)
citation_chunks = sum(1 for c in all_chunks if c.has_citation)
equation_chunks = sum(1 for c in all_chunks if c.has_equation)

print(f"\nFeature distribution:")
print(f"  Chunks with figure references: {figure_chunks}")
print(f"  Chunks with table references: {table_chunks}")
print(f"  Chunks with citations: {citation_chunks}")
print(f"  Chunks with equations: {equation_chunks}")


### Part 5: The Metadata Query Engine


In [ ]:
# We implement a simple filter system that mimics SQL WHERE clauses.
# Each filter is a callable that returns True/False for a chunk.

class MetadataFilter:
    """A single filter condition on chunk metadata."""

    def __init__(self, field_name, operator, value=None):
        self.field_name = field_name
        self.operator = operator
        self.value = value

    def apply(self, chunk):
        """Test whether a chunk passes this filter."""
        attr = getattr(chunk, self.field_name, None)
        if attr is None:
            return False
        if self.operator == "eq":
            return attr == self.value
        elif self.operator == "neq":
            return attr != self.value
        elif self.operator == "contains":
            return self.value.lower() in str(attr).lower()
        elif self.operator == "gt":
            return attr > self.value
        elif self.operator == "lt":
            return attr < self.value
        elif self.operator == "gte":
            return attr >= self.value
        elif self.operator == "lte":
            return attr <= self.value
        elif self.operator == "in":
            return attr in self.value
        elif self.operator == "is_true":
            return bool(attr) is True
        elif self.operator == "is_false":
            return bool(attr) is False
        return False

    def __repr__(self):
        return f"Filter({self.field_name} {self.operator} {self.value})"

class MetadataQueryEngine:
    """Run SQL-like queries against a collection of chunks.

    Usage:
        engine = MetadataQueryEngine(chunks)
        results = engine.query(
            filters=[MetadataFilter("page", "eq", 2)],
            sort_by="page",
            limit=10,
        )
    """

    def __init__(self, chunks):
        self.chunks = chunks

    def query(self, filters=None, sort_by=None, ascending=True, limit=None):
        """Apply filters, sort, and limit to the chunk collection.

        Args:
            filters: list of MetadataFilter objects (AND logic).
            sort_by: field name to sort results by.
            ascending: sort direction.
            limit: max number of results.

        Returns:
            List of ChunkMetadata objects matching all filters.
        """
        results = list(self.chunks)

        # Apply all filters (AND logic).
        if filters:
            for f in filters:
                results = [c for c in results if f.apply(c)]

        # Sort.
        if sort_by:
            results.sort(key=lambda c: getattr(c, sort_by, 0), reverse=not ascending)

        # Limit.
        if limit:
            results = results[:limit]

        return results

    def count(self, filters=None):
        """Count chunks matching the given filters."""
        return len(self.query(filters=filters))

engine = MetadataQueryEngine(all_chunks)
print("MetadataQueryEngine ready")


### Part 6: Simple Filter Queries


In [ ]:
# Start with single-condition filters.

# Query 1: All chunks on page 2.
results = engine.query(filters=[MetadataFilter("page", "eq", 2)])
print(f"Page == 2: {len(results)} chunks")
for c in results[:2]:
    preview = c.text[:60].replace("\n", " ")
    print(f"  [{c.section}] {preview}...")

# Query 2: Chunks in the Training section.
results = engine.query(filters=[MetadataFilter("section", "contains", "Training")])
print(f"\nSection contains 'Training': {len(results)} chunks")

# Query 3: Chunks with more than 80 words.
results = engine.query(filters=[MetadataFilter("word_count", "gt", 80)])
print(f"Word count > 80: {len(results)} chunks")


### Part 7: Compound Filter Queries (AND Logic)


In [ ]:
# Combine multiple filters to narrow results.

# Query 4: Chunks in Results section with table references.
results = engine.query(filters=[
    MetadataFilter("section", "contains", "Results"),
    MetadataFilter("has_table_ref", "is_true"),
])
print(f"Results section + has table ref: {len(results)} chunks")
for c in results:
    preview = c.text[:60].replace("\n", " ")
    print(f"  [Page {c.page}] {preview}...")

# Query 5: Chunks with citations, not in References section, on pages 6-9.
results = engine.query(filters=[
    MetadataFilter("has_citation", "is_true"),
    MetadataFilter("section", "neq", "References"),
    MetadataFilter("page", "gte", 6),
    MetadataFilter("page", "lte", 9),
])
print(f"\nCitations, non-References, pages 6-9: {len(results)} chunks")

# Query 6: Chunks with equations on pages 3-5 (architecture section).
results = engine.query(filters=[
    MetadataFilter("has_equation", "is_true"),
    MetadataFilter("page", "gte", 3),
    MetadataFilter("page", "lte", 5),
])
print(f"Equations on pages 3-5: {len(results)} chunks")


### Part 8: Sorted and Limited Queries


In [ ]:
# Combine filters with sorting and result limits.

# Top 5 longest chunks in the Model Architecture section.
results = engine.query(
    filters=[MetadataFilter("section", "contains", "Model Architecture")],
    sort_by="word_count",
    ascending=False,
    limit=5,
)
print("Top 5 longest chunks in Model Architecture:")
for c in results:
    print(f"  [Page {c.page}, {c.word_count} words] {c.text[:50].replace(chr(10), ' ')}...")


### Part 9: Aggregation Queries


In [ ]:
# Compute statistics over filtered results without returning the chunks.

class MetadataAggregator:
    """Compute aggregate statistics over query results."""

    def __init__(self, engine):
        self.engine = engine

    def avg_word_count(self, filters=None):
        results = self.engine.query(filters=filters)
        if not results:
            return 0
        return sum(c.word_count for c in results) / len(results)

    def count_by_section(self, filters=None):
        results = self.engine.query(filters=filters)
        counts = {}
        for c in results:
            counts[c.section] = counts.get(c.section, 0) + 1
        return dict(sorted(counts.items()))

    def count_by_feature(self, filters=None):
        results = self.engine.query(filters=filters)
        return {
            "figures": sum(1 for c in results if c.has_figure_ref),
            "tables": sum(1 for c in results if c.has_table_ref),
            "equations": sum(1 for c in results if c.has_equation),
            "citations": sum(1 for c in results if c.has_citation),
        }

agg = MetadataAggregator(engine)

# Average word count across all chunks.
avg = agg.avg_word_count()
print(f"Average word count per chunk: {avg:.1f}")

# Count by section for chunks with citations.
citation_dist = agg.count_by_section(
    filters=[MetadataFilter("has_citation", "is_true")]
)
print("\nCitation distribution by section:")
for sec, count in citation_dist.items():
    print(f"  {sec}: {count}")

# Feature count for chunks with equations.
eq_features = agg.count_by_feature(
    filters=[MetadataFilter("has_equation", "is_true")]
)
print(f"\nChunks with equations also have:")
for feat, count in eq_features.items():
    print(f"  {feat}: {count}")


### Part 10: SQL-like Query Builder


In [ ]:
# A more readable interface for building queries programmatically.

def select(chunks):
    """Start building a query (like SQL SELECT FROM)."""
    return QueryBuilder(chunks)

class QueryBuilder:
    """Fluent query builder for metadata filtering."""

    def __init__(self, chunks):
        self._chunks = chunks
        self._filters = []
        self._sort = None
        self._asc = True
        self._limit = None

    def where(self, field_name, operator, value=None):
        # `value` defaults to None so unary operators ("is_true", "is_false")
        # read naturally: .where("has_citation", "is_true")
        self._filters.append(MetadataFilter(field_name, operator, value))
        return self

    def order_by(self, field_name, ascending=True):
        self._sort = field_name
        self._asc = ascending
        return self

    def limit(self, n):
        self._limit = n
        return self

    def execute(self):
        engine = MetadataQueryEngine(self._chunks)
        return engine.query(
            filters=self._filters,
            sort_by=self._sort,
            ascending=self._asc,
            limit=self._limit,
        )

# SQL-like query: SELECT * FROM chunks WHERE section = 'Training' AND has_citation = true ORDER BY word_count DESC LIMIT 3
results = (
    select(all_chunks)
    .where("section", "contains", "Training")
    .where("has_citation", "is_true")
    .order_by("word_count", ascending=False)
    .limit(3)
    .execute()
)
print("SQL-like query: Training section, with citations, top 3 by word count:")
for c in results:
    print(f"  [Page {c.page}, {c.word_count} words] {c.text[:50].replace(chr(10), ' ')}...")


### Part 11: Export Metadata as JSON


In [ ]:
# Persist the metadata index for reuse across sessions.

metadata_export = []
for c in all_chunks:
    d = asdict(c)
    d.pop("text", None)  # Exclude full text from the index.
    metadata_export.append(d)

export_path = Path(PDF_PATH).parent / "transformer_metadata_index.json"
with open(export_path, "w", encoding="utf-8") as f:
    json.dump(metadata_export, f, indent=2)

print(f"Exported {len(metadata_export)} metadata entries to {export_path.name}")


### Part 12: Summary


In [ ]:
# Structured metadata filtering gives us SQL-like power without vectors:
#
# 1. Rich metadata per chunk: page, section, word count, features.
# 2. Compound filters with AND logic for precise retrieval.
# 3. Aggregation queries for statistics over filtered sets.
# 4. Fluent query builder for readable programmatic queries.
# 5. JSON export for persistent metadata indices.
#
# This approach works exceptionally well for documents with clear structure.
# Combined with page-based retrieval from Step 1, it forms a complete
# vectorless retrieval system.

print("Structured Metadata Filtering Summary:")
print("  Attach 14 metadata fields to each text chunk")
print("  Filter by any field with SQL-like operators")
print("  Combine filters with AND logic")
print("  Sort, limit, and aggregate results")
print("  Export metadata index as JSON for reuse")
print()
print("Next: Input/Output Guardrails")
